# MLP — Brain Tumour MRI Classification

**Owner:** Member 1 — Siddhant Bagchi (230953260)  
**Project:** A Comparative Study of Deep Learning Architectures for Brain Tumour MRI Classification (ICT-4442)

Classical baseline: flattened pixels → 3 fully-connected layers. No convolution, no attention.

All four models share the same data split, preprocessing and training/evaluation code (`common/`); only the architecture and learning rate differ.

> The printed outputs below are from the original run on a Kaggle GPU. The last cell writes the files the comparison scripts in `analysis/` read.

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root, so `common` and `models` can be imported

import torch
from common.config import CLASSES, OUTPUT_DIR, device, set_seed
from common.data import describe_dataset, get_loaders
from common.engine import train_model, evaluate_model, get_probs_and_labels
from common.results import save_run

print(f"Device: {device}")
set_seed()

Device: cuda


## 2. Data (shared split)

In [2]:
describe_dataset()

Train classes found: ['glioma', 'meningioma', 'notumor', 'pituitary']
Test classes found:  ['glioma', 'meningioma', 'notumor', 'pituitary']
  glioma: 1400 images
  meningioma: 1400 images
  notumor: 1400 images
  pituitary: 1400 images


In [3]:
train_loader, val_loader, test_loader = get_loaders()

Train: 5040 | Val: 560 | Test: 1600


## 3. Model — MLP

In [4]:
from models.mlp import MLPBaseline

n_params = sum(p.numel() for p in MLPBaseline().parameters())
print(f"MLP parameters: {n_params:,}")

MLP parameters: 77,137,028


## 4. Train (10 epochs, Adam, cross-entropy) and evaluate on the test set

In [5]:
mlp_model = MLPBaseline(num_classes=len(CLASSES))
mlp_model, mlp_history = train_model(mlp_model, train_loader, val_loader, epochs=10, model_name="MLP")

mlp_loss, mlp_acc, mlp_prec, mlp_rec, mlp_f1, mlp_cm = evaluate_model(mlp_model, test_loader)
print(f"\nMLP Test Results — Acc: {mlp_acc:.4f}  Precision: {mlp_prec:.4f}  Recall: {mlp_rec:.4f}  F1: {mlp_f1:.4f}")

torch.save(mlp_model.state_dict(), OUTPUT_DIR / 'mlp_final.pth')

[MLP] Epoch 1/10  Train Loss: 1.3600  Train Acc: 0.5226  Val Loss: 0.9014  Val Acc: 0.6500
[MLP] Epoch 2/10  Train Loss: 0.9750  Train Acc: 0.5968  Val Loss: 0.7835  Val Acc: 0.6946
[MLP] Epoch 3/10  Train Loss: 0.8735  Train Acc: 0.6347  Val Loss: 0.7322  Val Acc: 0.7179
[MLP] Epoch 4/10  Train Loss: 0.8365  Train Acc: 0.6615  Val Loss: 0.7107  Val Acc: 0.7393
[MLP] Epoch 5/10  Train Loss: 0.8313  Train Acc: 0.6563  Val Loss: 0.7315  Val Acc: 0.7000
[MLP] Epoch 6/10  Train Loss: 0.8403  Train Acc: 0.6383  Val Loss: 0.6549  Val Acc: 0.7464
[MLP] Epoch 7/10  Train Loss: 0.7734  Train Acc: 0.6758  Val Loss: 0.6915  Val Acc: 0.7304
[MLP] Epoch 8/10  Train Loss: 0.7554  Train Acc: 0.6859  Val Loss: 0.6722  Val Acc: 0.7196
[MLP] Epoch 9/10  Train Loss: 0.7538  Train Acc: 0.6935  Val Loss: 0.6335  Val Acc: 0.7679
[MLP] Epoch 10/10  Train Loss: 0.7256  Train Acc: 0.6970  Val Loss: 0.6530  Val Acc: 0.7429

MLP Test Results — Acc: 0.7188  Precision: 0.7215  Recall: 0.7188  F1: 0.7187


## 5. Save results for the comparison scripts

In [6]:
save_run("mlp", mlp_model, mlp_history, test_loader)